# 128-DWT（3線形補間）カリキュラム事前学習

比較対象の通常128モデルと出力解像度を揃える。処理は `128×256×256 → 3線形補間 → 64×128×128 → 3D Haar DWT → 8成分 (32×64×64) → 既存ネットワーク内の4群化 → 全8成分を同一DVFでwarp → inverse DWT → 64×128×128`。

カリキュラムの ±1〜±40 px は、**再構成後の `64×128×128` 画像のピクセル単位**である。係数格子は各軸半分のため、DVF教師値は係数格子へ落とす際に `/2` している。

In [ ]:
from pathlib import Path
import sys
import torch

# ノートブックを Saito フォルダから実行する想定。voxelmorph の親を import path に追加する。
PROJECT_ROOT = Path(r'D:\Saito')
if not (PROJECT_ROOT / 'voxelmorph').is_dir():
    raise FileNotFoundError(f'voxelmorph folder was not found under: {PROJECT_ROOT}')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dwt128_trilinear_curriculum import TARGET_SHAPE, COEFFICIENT_SHAPE, run_curriculum_pretraining

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DATA_PATH = Path('Data/TrainData_NoBed.npz')
CHECKPOINT_DIR = Path('128dwt_trilinear_curriculum_checkpoints')

print('device:', device)
print('target image:', TARGET_SHAPE)
print('DWT coefficients:', COEFFICIENT_SHAPE)
print('training data:', DATA_PATH.resolve())
print('checkpoints:', CHECKPOINT_DIR.resolve())

In [ ]:
# 256版と同じカリキュラム: 1 stage = 2,000 epoch、1→40 px、計80,000 epoch。
TOTAL_EPOCHS = 80_000
STAGE_EPOCHS = 2_000
RUN_TRAINING = True  # 実行前に True へ変更

# ここは事前学習なので None のまま。
# 別患者fine-tune後に同じ ±1→±40 学習を追加で行う場合だけ、
# その fine-tune 最終重みへのパスを指定し、prefix/output_dir を別名にする。
INITIAL_CHECKPOINT = None
CHECKPOINT_PREFIX = '128dwt_trilinear_curriculum'

if RUN_TRAINING:
    model_dwt, final_path, history = run_curriculum_pretraining(
        data_path=DATA_PATH,
        output_dir=CHECKPOINT_DIR,
        total_epochs=TOTAL_EPOCHS,
        stage_epochs=STAGE_EPOCHS,
        batch_size=2,
        initial_checkpoint=INITIAL_CHECKPOINT,
        checkpoint_prefix=CHECKPOINT_PREFIX,
        device=device,
    )
    print('final checkpoint:', final_path.resolve())
else:
    print('RUN_TRAINING=False: 設定確認のみ。True にしてから実行する。')

In [ ]:
import inspect
import dwt128_trilinear_curriculum as curriculum

source = inspect.getsource(curriculum.run_curriculum_pretraining)
lines = source.splitlines()

for line_no, line in enumerate(lines, start=1):
    if (
        'model(' in line
        or 'haar_analysis_3d' in line
        or 'resize_to_target' in line
        or 'SpatialTransformer' in line
        or 'image_loss' in line
        or 'flow_loss' in line
        or 'smoothness' in line
    ):
        start = max(1, line_no - 6)
        end = min(len(lines), line_no + 10)
        print(f'\n--- lines {start}-{end} ---')
        print('\n'.join(lines[start - 1:end]))

In [ ]:
# ============================================================
# 80,000 epoch カリキュラム重み
#   → TrainData_NoBed.npz で別患者 fine-tuning（30,000 epoch）
# ============================================================

from pathlib import Path
import dwt128_trilinear_curriculum as curriculum

# カリキュラム学習で保存された「80,000 epoch」重みを明示指定
CURRICULUM_CHECKPOINT = (
    PROJECT_ROOT
    / '128dwt_trilinear_curriculum_checkpoints'
    / '128dwt_trilinear_curriculum_1to40_final.pth'
)

TRAIN_DATA_PATH = PROJECT_ROOT / 'Data' / 'TrainData_NoBed.npz'

FINETUNE_OUTPUT_DIR = (
    PROJECT_ROOT / '128dwt_trilinear_different_patients_finetune_checkpoints'
)

FINETUNE_EPOCHS = 30_000
FINETUNE_STAGE_EPOCHS = 2_000
RUN_FINETUNING = True

if not CURRICULUM_CHECKPOINT.exists():
    raise FileNotFoundError(
        f'80,000 epochのカリキュラム重みがありません:\n{CURRICULUM_CHECKPOINT}'
    )

if not TRAIN_DATA_PATH.exists():
    raise FileNotFoundError(f'TrainData_NoBed.npz がありません:\n{TRAIN_DATA_PATH}')

# 既存の別患者fine-tune関数を TrainData_NoBed.npz に対応させる
original_loader = curriculum.load_different_patient_volumes

def load_train_npz_as_different_patients(data_path, key='Train'):
    volumes = curriculum.load_training_volumes(data_path, key=key)

    if len(volumes) < 2:
        raise ValueError('別患者fine-tuningには2患者以上が必要です。')

    source_names = [
        f'{Path(data_path).name}:patient_{i:03d}'
        for i in range(len(volumes))
    ]

    print('TrainData_NoBed volumes:', volumes.shape)
    return volumes, source_names

curriculum.load_different_patient_volumes = load_train_npz_as_different_patients

print('80k curriculum checkpoint:', CURRICULUM_CHECKPOINT.resolve())
print('fine-tuning data:', TRAIN_DATA_PATH.resolve())
print('fine-tuning output:', FINETUNE_OUTPUT_DIR.resolve())

try:
    if RUN_FINETUNING:
        finetune_model, finetune_final_path, finetune_history = (
            curriculum.run_different_patient_finetuning(
                data_root=TRAIN_DATA_PATH,
                pretrained_checkpoint=CURRICULUM_CHECKPOINT,
                output_dir=FINETUNE_OUTPUT_DIR,
                total_epochs=FINETUNE_EPOCHS,
                stage_epochs=FINETUNE_STAGE_EPOCHS,
                batch_size=2,
                learning_rate=1e-6,
                image_weight=100.0,
                flow_weight=0.0,
                smoothness_weight=0.1,
                feature_channels=16,
                checkpoint_prefix='128dwt_trilinear_different_patients_finetune',
                visualization_every=2_000,
                device=device,
            )
        )

        print('110k相当の最終重み:', finetune_final_path.resolve())

finally:
    curriculum.load_different_patient_volumes = original_loader

In [ ]:
from pathlib import Path
import csv
import importlib
import matplotlib.pyplot as plt
import numpy as np
import torch

import dwt128_trilinear_curriculum as curriculum
importlib.reload(curriculum)

DATA_ROOT = PROJECT_ROOT / 'Data' / 'TestData'

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / '128dwt_trilinear_different_patients_finetune_checkpoints'
    / '128dwt_trilinear_different_patients_finetune_final.pth'
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / '128dwt_trilinear_different_patients_test_evaluation'
)
PREVIEW_DIR = OUTPUT_DIR / 'previews'

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'TestData がありません:\n{DATA_ROOT}')

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'最終checkpointがありません:\n{CHECKPOINT_PATH}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(exist_ok=True)

# checkpointからモデル設定を読み、80k + 30k fine-tune済み重みをロード
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
feature_channels = int(checkpoint.get('feature_channels', 16))

model = curriculum.make_model(device, feature_channels=feature_channels)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print('test data :', DATA_ROOT.resolve())
print('checkpoint:', CHECKPOINT_PATH.resolve())
print('output    :', OUTPUT_DIR.resolve())
print('feature channels:', feature_channels)

def calculate_metrics(fixed, moved):
    error = fixed - moved
    mse = float(np.mean(error ** 2))
    fixed_zero = fixed - fixed.mean()
    moved_zero = moved - moved.mean()

    return {
        'mse': mse,
        'rmse': float(np.sqrt(mse)),
        'mae': float(np.mean(np.abs(error))),
        'ncc': float(
            np.mean(fixed_zero * moved_zero)
            / (fixed_zero.std() * moved_zero.std() + 1e-8)
        ),
    }

def save_preview(moving, fixed, moved, output_path, title):
    z = fixed.shape[0] // 2
    difference = np.abs(fixed - moved)

    vmin, vmax = np.percentile(
        np.concatenate((moving.ravel(), fixed.ravel())),
        (1, 99),
    )

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    for axis, image, name in zip(
        axes[:3],
        (moving, fixed, moved),
        ('Moving', 'Fixed', 'Moved'),
    ):
        axis.imshow(image[z], cmap='gray', vmin=vmin, vmax=vmax)
        axis.set_title(name)
        axis.axis('off')

    axes[3].imshow(difference[z], cmap='magma')
    axes[3].set_title('|Fixed − Moved|')
    axes[3].axis('off')

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

pair_dirs = sorted(path for path in DATA_ROOT.iterdir() if path.is_dir())

if not pair_dirs:
    raise FileNotFoundError(
        f'{DATA_ROOT} 内にペア用フォルダがありません。'
    )

results = []

with torch.no_grad():
    for pair_dir in pair_dirs:
        npz_files = sorted(pair_dir.glob('*.npz'))

        if len(npz_files) != 2:
            print(f'Skip: {pair_dir.name} (.npzが2個ではない)')
            continue

        # ファイル名の昇順で Moving / Fixed を決める
        moving_volumes = curriculum.load_training_volumes(npz_files[0])
        fixed_volumes = curriculum.load_training_volumes(npz_files[1])

        count = min(len(moving_volumes), len(fixed_volumes))

        for index in range(count):
            moving = torch.from_numpy(
                moving_volumes[index:index + 1]
            ).unsqueeze(1).to(device)

            fixed = torch.from_numpy(
                fixed_volumes[index:index + 1]
            ).unsqueeze(1).to(device)

            # 128×256×256 → 64×128×128 → DWT → 位置合わせ → Synthesis
            moving_target = curriculum.resize_to_target(moving)
            fixed_target = curriculum.resize_to_target(fixed)

            moved, _, predicted_flow = model(
                curriculum.haar_analysis_3d(moving_target),
                curriculum.haar_analysis_3d(fixed_target),
            )

            moving_np = moving_target[0, 0].cpu().numpy()
            fixed_np = fixed_target[0, 0].cpu().numpy()
            moved_np = moved[0, 0].cpu().numpy()

            row = calculate_metrics(fixed_np, moved_np)
            row.update({
                'pair': pair_dir.name,
                'volume_index': index,
                'moving_file': npz_files[0].name,
                'fixed_file': npz_files[1].name,
                'mean_flow_magnitude': float(
                    torch.linalg.vector_norm(predicted_flow, dim=1).mean().cpu()
                ),
            })

            results.append(row)

            save_preview(
                moving_np,
                fixed_np,
                moved_np,
                PREVIEW_DIR / f'{pair_dir.name}_volume_{index:03d}.png',
                f'{pair_dir.name} / volume {index} | '
                f'RMSE={row["rmse"]:.5f}, NCC={row["ncc"]:.4f}',
            )

            print(
                f'{pair_dir.name} volume {index}: '
                f'RMSE={row["rmse"]:.6f}, NCC={row["ncc"]:.4f}'
            )

if not results:
    raise RuntimeError('評価できる別患者ペアが見つかりませんでした。')

with (OUTPUT_DIR / 'per_volume_metrics.csv').open(
    'w', newline='', encoding='utf-8'
) as file:
    writer = csv.DictWriter(file, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

summary = {
    'n_volumes': len(results),
    **{
        key: float(np.mean([row[key] for row in results]))
        for key in ('mse', 'rmse', 'mae', 'ncc', 'mean_flow_magnitude')
    },
}

with (OUTPUT_DIR / 'summary_metrics.csv').open(
    'w', newline='', encoding='utf-8'
) as file:
    writer = csv.DictWriter(file, fieldnames=summary.keys())
    writer.writeheader()
    writer.writerow(summary)

print('\n========== Test evaluation summary ==========')
print(summary)

results

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import glob
from skimage.metrics import structural_similarity as ssim
from tabulate import tabulate
from piq import fsim

# ======================
# 定数（ここで評価したいスライス範囲を指定）
# ======================
CT_MIN, CT_MAX = -1200.0, 3146.98
SLICE_SIZE = (64, 128, 128)
SLICE_IDX_VIEW = 32    # デフォルト表示スライス（範囲指定がある場合は中央スライスが使われる）
SLICE_IDX_FSIM = 28

# 評価したいスライス範囲を指定（例: 20〜40）
# None にすると全ボリュームで評価します
SLICE_START = None
SLICE_END = None

# ======================
# 評価関数（変更なし）
# ======================
def compute_rmse(vol1, vol2):
    return np.sqrt(np.mean((vol1 - vol2) ** 2))

import torch
import torch.nn.functional as F
import numpy as np

def _gaussian_kernel_3d(window_size=7, sigma=1.5, device='cpu', dtype=torch.float32):
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    g1 = torch.exp(-(coords**2) / (2 * sigma**2))
    g1 = g1 / g1.sum()
    g3 = g1[:, None, None] * g1[None, :, None] * g1[None, None, :]
    return g3.unsqueeze(0).unsqueeze(0)  # shape (1,1,D,H,W)

def _ssim_components_3d(x, y, window, K1=0.01, K2=0.03, eps=1e-12):
    """
    Compute luminance, contrast, structure maps for 3D volumes.
    x,y : tensors with shape (N,1,D,H,W)
    window : gaussian kernel shape (1,1,d,h,w)
    returns (l_map, cs_map) where cs_map = contrast * structure
    """
    pad = tuple([s//2 for s in window.shape[-3:]])
    mu_x = F.conv3d(x, window, padding=pad)
    mu_y = F.conv3d(y, window, padding=pad)

    mu_x_sq = mu_x * mu_x
    mu_y_sq = mu_y * mu_y
    mu_xy = mu_x * mu_y

    sigma_x_sq = F.conv3d(x * x, window, padding=pad) - mu_x_sq
    sigma_y_sq = F.conv3d(y * y, window, padding=pad) - mu_y_sq
    sigma_xy = F.conv3d(x * y, window, padding=pad) - mu_xy

    sigma_x_sq = torch.clamp(sigma_x_sq, min=0.0)
    sigma_y_sq = torch.clamp(sigma_y_sq, min=0.0)

    sigma_x = torch.sqrt(sigma_x_sq + eps)
    sigma_y = torch.sqrt(sigma_y_sq + eps)

    # Estimate L (data range) per-sample for numeric C's
    N = x.shape[0]
    max_x = x.view(N, -1).max(dim=1)[0].view(N,1,1,1,1)
    min_x = x.view(N, -1).min(dim=1)[0].view(N,1,1,1,1)
    max_y = y.view(N, -1).max(dim=1)[0].view(N,1,1,1,1)
    min_y = y.view(N, -1).min(dim=1)[0].view(N,1,1,1,1)
    L = torch.max(max_x - min_x, max_y - min_y)
    L = torch.clamp(L, min=eps)

    C1 = (K1 * L) ** 2
    C2 = (K2 * L) ** 2
    C3 = C2 / 2.0

    l_map = (2.0 * mu_xy + C1) / (mu_x_sq + mu_y_sq + C1 + eps)
    contrast = (2.0 * sigma_x * sigma_y + C2) / (sigma_x_sq + sigma_y_sq + C2 + eps)
    structure = (sigma_xy + C3) / (sigma_x * sigma_y + C3 + eps)

    cs_map = contrast * structure
    return l_map.clamp(min=eps), cs_map.clamp(min=eps)

def ms_ssim_3d(vol1, vol2,
               window_size=7, sigma=1.5,
               levels=5,
               weights=None,
               K1=0.01, K2=0.03, eps=1e-12,
               device=None):
    """
    Multi-scale SSIM for 3D volumes.
    vol1, vol2 : torch tensor or numpy array, shapes supported:
        (N,C,D,H,W), (C,D,H,W), (D,H,W)
    levels : number of scales (default 5)
    weights : list/tuple of length `levels` of weights summing to 1 (if None, use default MS-SSIM weights)
    returns: python float (mean MS-SSIM over batch)
    """
    # default MS-SSIM weights from the original paper (for 5 scales)
    if weights is None:
        # note: these are standard for 5-scale MS-SSIM
        weights = [0.0448, 0.2856, 0.3001, 0.2363, 0.1333]
    if len(weights) != levels:
        raise ValueError("len(weights) must equal levels")

    # convert numpy -> torch
    is_numpy = isinstance(vol1, np.ndarray) or isinstance(vol2, np.ndarray)
    if is_numpy:
        vol1 = torch.from_numpy(np.array(vol1))
        vol2 = torch.from_numpy(np.array(vol2))

    if not torch.is_tensor(vol1) or not torch.is_tensor(vol2):
        raise TypeError("vol1/vol2 must be numpy or torch tensor")

    # determine device
    if device is None:
        device = vol1.device if hasattr(vol1, 'device') else torch.device('cpu')
    device = torch.device(device)

    vol1 = vol1.to(device=device, dtype=torch.float32)
    vol2 = vol2.to(device=device, dtype=torch.float32)

    # normalize shapes to (N,C,D,H,W)
    def _ensure5d(x):
        if x.dim() == 5:
            return x
        if x.dim() == 4:
            return x.unsqueeze(0)
        if x.dim() == 3:
            return x.unsqueeze(0).unsqueeze(0)
        raise ValueError("Unsupported tensor shape: {}".format(x.shape))

    x = _ensure5d(vol1)
    y = _ensure5d(vol2)

    # unify channels by mean (if needed)
    if x.shape[1] != y.shape[1] or x.shape[1] > 1:
        x = x.mean(dim=1, keepdim=True)
        y = y.mean(dim=1, keepdim=True)

    N, C, D, H, W = x.shape

    # create gaussian kernel once
    window = _gaussian_kernel_3d(window_size=window_size, sigma=sigma, device=device, dtype=x.dtype)

    mcs = []   # mean contrast-structure per scale
    for lvl in range(levels):
        # if volume becomes too small to downsample, stop early
        if min(D, H, W) < 2:
            # compute final scale components and break
            l_map, cs_map = _ssim_components_3d(x, y, window, K1=K1, K2=K2, eps=eps)
            mcs.append(cs_map.view(N, -1).mean(dim=1))  # shape (N,)
            break

        l_map, cs_map = _ssim_components_3d(x, y, window, K1=K1, K2=K2, eps=eps)
        # mean over spatial dims gives per-sample scalar
        mcs.append(cs_map.view(N, -1).mean(dim=1))

        # downsample for next scale using average pooling (anti-aliasing)
        # kernel_size=2, stride=2
        x = F.avg_pool3d(x, kernel_size=2, stride=2, padding=0)
        y = F.avg_pool3d(y, kernel_size=2, stride=2, padding=0)
        N, C, D, H, W = x.shape

    # last scale luminance
    l_map, cs_map = _ssim_components_3d(x, y, window, K1=K1, K2=K2, eps=eps)
    l_mean = l_map.view(N, -1).mean(dim=1)  # per-sample

    # If we generated fewer mcs than requested levels (due to small volume), adjust weights
    actual_levels = len(mcs)
    if actual_levels < levels:
        # use last `actual_levels` weights and normalized
        w = torch.tensor(weights[:actual_levels], device=device, dtype=x.dtype)
        w = w / w.sum()
        used_weights = w
        # last weight for luminance is taken as original last weight mapped proportionally
        lum_weight = weights[min(actual_levels-1, len(weights)-1)]
    else:
        used_weights = torch.tensor(weights[:levels-1], device=device, dtype=x.dtype)  # weights for cs scales except last
        lum_weight = weights[levels-1]

    # compute MS-SSIM per sample:
    # product over scales of (mcs_i ^ weight_i)  and multiply by (l_mean ^ lum_weight)
    # convert list of tensors to shape (actual_levels, N)
    mcs_t = torch.stack(mcs[:actual_levels], dim=0)  # shape (actual_levels, N)
    # choose weights for these mcs: if actual_levels == levels -> first (levels-1) are cs weights, last is cs too
    if actual_levels == levels:
        cs_weights = torch.tensor(weights[:levels-1], device=device, dtype=x.dtype)
        # there are levels-1 cs entries and last is l_mean
        # mcs_t has length levels-1 (we appended cs for each scale before downsample), plus we still have final cs? 
        # Here we've appended cs at each level before downsampling, and computed final l_map after last downsample.
        # For typical MS-SSIM: use cs means from all levels and l only from last one.
        # So mcs_t currently includes cs for all levels (length == levels). In our loop we appended cs each iteration, and after final downsample we appended final cs too.
        # To match standard weights: use weights[:levels] for cs and lum_weight for l.
        cs_weights = torch.tensor(weights[:actual_levels], device=device, dtype=x.dtype)
    else:
        # when smaller, distribute weights proportionally — use used_weights for cs
        cs_weights = used_weights

    # Ensure cs_weights length matches mcs_t length
    if cs_weights.numel() != mcs_t.shape[0]:
        # simple fallback: evenly distribute
        cs_weights = torch.ones(mcs_t.shape[0], device=device, dtype=x.dtype) / float(mcs_t.shape[0])

    # raise mcs to weights and multiply (per-sample)
    # mcs_t ** cs_weights[:,None] -> shape (L, N)
    # product over rows -> (N,)
    ms_prod = torch.prod(mcs_t.pow(cs_weights.view(-1,1)), dim=0)
    ms_l = l_mean.pow(float(lum_weight))
    ms_per_sample = ms_prod * ms_l

    # return mean over batch as python float
    return float(ms_per_sample.mean().item())

def compute_fsim(vol1, vol2, device):
    """中央スライスでFSIMを評価（ただしこのスクリプトでは通常中央は範囲中央値に合わせる）"""
    v1 = vol1
    v2 = vol2
    # v1, v2 expected shape (C, D, H, W) or (D, H, W). Normalize access:
    if np.asarray(v1).ndim == 3:
        D = v1.shape[0]
    else:
        D = np.asarray(v1).shape[1]
    idx = min(SLICE_IDX_FSIM, max(0, D - 1))
    # Build tensors robustly for either shape
    if np.asarray(v1).ndim == 3:
        fixed_tensor = torch.tensor(v1[idx]).unsqueeze(0).unsqueeze(0).float().to(device)
        pred_tensor = torch.tensor(v2[idx]).unsqueeze(0).unsqueeze(0).float().to(device)
    else:
        fixed_tensor = torch.tensor(v1[:, idx, :, :]).unsqueeze(1).float().to(device)
        pred_tensor = torch.tensor(v2[:, idx, :, :]).unsqueeze(1).float().to(device)
        fixed_tensor = fixed_tensor[0].unsqueeze(0)
        pred_tensor = pred_tensor[0].unsqueeze(0)
    try:
        return fsim(pred_tensor, fixed_tensor, data_range=1.0, chromatic=False).item()
    except Exception:
        return float('nan')

def compute_ncc(vol1, vol2):
    v1 = vol1.flatten().astype(np.float32)
    v2 = vol2.flatten().astype(np.float32)
    v1_mean = v1.mean()
    v2_mean = v2.mean()
    numerator = np.sum((v1 - v1_mean) * (v2 - v2_mean))
    denominator = np.sqrt(np.sum((v1 - v1_mean) ** 2) * np.sum((v2 - v2_mean) ** 2) + 1e-8)
    return numerator / denominator

def compute_metrics(fixed, transformed, device, model_name):
    """Dice, Jaccard, SSIM, FSIM, RMSE, NCC をまとめて計算（入力は部分ボリュームでも可）"""
    fixed_bin = (fixed > 0.144).astype(int)
    transformed_bin = (transformed > 0.144).astype(int)

    dice = 2.0 * np.logical_and(fixed_bin, transformed_bin).sum() / (fixed_bin.sum() + transformed_bin.sum() + 1e-8)
    jaccard = np.logical_and(fixed_bin, transformed_bin).sum() / (np.logical_or(fixed_bin, transformed_bin).sum() + 1e-8)
#     ssim_val = compute_ssim_3d(fixed, transformed)
    ssim_val = ms_ssim_3d(fixed, transformed, levels=5, device='cpu')
    fsim_val = compute_fsim(fixed, transformed, device)
    ncc_val = compute_ncc(fixed, transformed)

    # RMSE用にCT値を復元
    pred_denorm = transformed * (CT_MAX - CT_MIN) + CT_MIN
    true_denorm = fixed * (CT_MAX - CT_MIN) + CT_MIN
    rmse_val = compute_rmse(true_denorm, pred_denorm)

    return [model_name, dice, jaccard, ssim_val, fsim_val, rmse_val, ncc_val]

# ======================
# 表示ユーティリティ（どちらの形でも扱えるようにする）
# ======================
def _ensure_channel_first_np(vol):
    v = np.asarray(vol)
    if v.ndim == 3:
        return v[np.newaxis, ...]
    if v.ndim == 4:
        return v
    raise ValueError(f"Unexpected array shape {v.shape} in _ensure_channel_first_np")

def show_images(moving, fixed, transformed_images, slice_idx=None):
    """
    display grid; slice_idx: if None, use central slice of available depth
    """
    mv = _ensure_channel_first_np(moving)
    fx = _ensure_channel_first_np(fixed)
    # choose slice idx
    if slice_idx is None:
        slice_idx = min(mv.shape[1], fx.shape[1]) // 2
    slice_idx = max(0, min(slice_idx, min(mv.shape[1], fx.shape[1]) - 1))

    imgs = []
    imgs.append((mv[0, slice_idx], "Moving Image"))
    imgs.append((fx[0, slice_idx], "Fixed Image"))
    for name, img in transformed_images.items():
        v = _ensure_channel_first_np(img)
        sidx = min(slice_idx, v.shape[1] - 1)
        imgs.append((v[0, sidx], name))

    n = len(imgs)
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax, (img2d, title) in zip(axes, imgs):
        ax.imshow(img2d, cmap="gray")
        ax.set_title(title)
        ax.axis('off')
    for ax in axes[len(imgs):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
import importlib
from tabulate import tabulate
import numpy as np
import torch

import dwt128_trilinear_curriculum as curriculum
importlib.reload(curriculum)

DATA_ROOT = PROJECT_ROOT / 'Data' / 'TestData'

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / '128dwt_trilinear_different_patients_finetune_checkpoints'
    / '128dwt_trilinear_different_patients_finetune_final.pth'
)

MODEL_NAME = 'Curriculum-80k + Different-patient fine-tune-30k'

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'checkpointがありません:\n{CHECKPOINT_PATH}')

# 最終モデルを読み込む
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
feature_channels = int(checkpoint.get('feature_channels', 16))

model = curriculum.make_model(device, feature_channels=feature_channels)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

pair_dirs = sorted(path for path in DATA_ROOT.iterdir() if path.is_dir())

if not pair_dirs:
    raise FileNotFoundError(f'評価ペアフォルダがありません:\n{DATA_ROOT}')

all_results = []

print('checkpoint:', CHECKPOINT_PATH.resolve())
print('test data :', DATA_ROOT.resolve())

with torch.no_grad():
    for pair_dir in pair_dirs:
        npz_files = sorted(pair_dir.glob('*.npz'))

        if len(npz_files) != 2:
            print(f'Skip: {pair_dir.name}（npzが2個ではありません）')
            continue

        # ファイル名順で Moving / Fixed を決定
        moving_volumes = curriculum.load_training_volumes(npz_files[0])
        fixed_volumes = curriculum.load_training_volumes(npz_files[1])

        volume_count = min(len(moving_volumes), len(fixed_volumes))

        for volume_index in range(volume_count):
            moving = torch.from_numpy(
                moving_volumes[volume_index:volume_index + 1]
            ).unsqueeze(1).to(device)

            fixed = torch.from_numpy(
                fixed_volumes[volume_index:volume_index + 1]
            ).unsqueeze(1).to(device)

            # 128×256×256 → 64×128×128 → DWT → DVF推定・warp → Synthesis
            moving_target = curriculum.resize_to_target(moving)
            fixed_target = curriculum.resize_to_target(fixed)

            moved, _, predicted_flow = model(
                curriculum.haar_analysis_3d(moving_target),
                curriculum.haar_analysis_3d(fixed_target),
            )

            moving_np = moving_target[0, 0].cpu().numpy()
            fixed_np = fixed_target[0, 0].cpu().numpy()
            moved_np = moved[0, 0].cpu().numpy()

            # 貼ってくれたコードと同じ指標:
            # Dice / Jaccard / MS-SSIM / FSIM / RMSE / NCC
            values = compute_metrics(
                fixed_np,
                moved_np,
                device=device,
                model_name=MODEL_NAME,
            )

            row = {
                'pair': pair_dir.name,
                'volume': volume_index,
                'model': values[0],
                'dice': values[1],
                'jaccard': values[2],
                'ms_ssim': values[3],
                'fsim': values[4],
                'rmse_hu': values[5],
                'ncc': values[6],
                'mean_flow_magnitude': float(
                    torch.linalg.vector_norm(predicted_flow, dim=1).mean().cpu()
                ),
            }
            all_results.append(row)

            print(f'\n===== {pair_dir.name} / volume {volume_index} =====')
            print(tabulate(
                [[
                    f'{row["dice"]:.4f}',
                    f'{row["jaccard"]:.4f}',
                    f'{row["ms_ssim"]:.4f}',
                    f'{row["fsim"]:.4f}',
                    f'{row["rmse_hu"]:.2f}',
                    f'{row["ncc"]:.4f}',
                    f'{row["mean_flow_magnitude"]:.4f}',
                ]],
                headers=['Dice', 'Jaccard', 'MS-SSIM', 'FSIM', 'RMSE [HU]', 'NCC', 'Mean |DVF|'],
                tablefmt='github',
            ))

            show_images(
                moving_np,
                fixed_np,
                {'Moved Image': moved_np},
                slice_idx=SLICE_IDX_VIEW,
            )

if not all_results:
    raise RuntimeError('評価できるTestDataペアがありませんでした。')

print('\n========== 全テストペアの平均 ==========')

mean_metrics = {
    key: float(np.nanmean([row[key] for row in all_results]))
    for key in (
        'dice',
        'jaccard',
        'ms_ssim',
        'fsim',
        'rmse_hu',
        'ncc',
        'mean_flow_magnitude',
    )
}

print(tabulate(
    [[
        f'{mean_metrics["dice"]:.4f}',
        f'{mean_metrics["jaccard"]:.4f}',
        f'{mean_metrics["ms_ssim"]:.4f}',
        f'{mean_metrics["fsim"]:.4f}',
        f'{mean_metrics["rmse_hu"]:.2f}',
        f'{mean_metrics["ncc"]:.4f}',
        f'{mean_metrics["mean_flow_magnitude"]:.4f}',
    ]],
    headers=['Dice', 'Jaccard', 'MS-SSIM', 'FSIM', 'RMSE [HU]', 'NCC', 'Mean |DVF|'],
    tablefmt='github',
))

all_results